# 3. Random Forest
**Use case:** Insurance — Auto insurance claims fraud detection (SIU referral)

Industry estimates commonly put insurance fraud at around 10% of property-casualty losses, and auto claims are a frequent target: staged accidents, inflated repair bills and padded injury claims. Honest customers pay through higher premiums, while the SIU has only a handful of investigators who cannot review every claim. The insurer has claim details, policy tenure, police reports, witnesses, repair shop data and prior claims. Success means putting the most suspicious claims at the top of the SIU queue so investigators find more fraud per hour while honest claims are paid quickly.

> 💡 Instead of trusting one claims adjuster, you ask a hundred adjusters, each of whom has seen a different random pile of past claims and only looks at a few facts at a time. Their combined vote is far more reliable than any single opinion.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: Three trees vote on one suspicious claim
Train a tiny Random Forest on 14 hand-typed auto claims, inspect each tree's vote and see the forest average them.

| 5W + H | |
|---|---|
| **Who** | A claims manager curious how the SIU referral score is produced. |
| **What** | A forest that votes on whether a new auto claim should be referred to the Special Investigations Unit (SIU). |
| **When** | At first notice of loss (FNOL), when the adjuster opens the claim. |
| **Where** | In the claims system's triage step, before payment is authorised. |
| **Why** | Seeing individual trees disagree and then combine makes 'wisdom of the crowd' concrete. |
| **How** | Store 14 claims in a dict of lists, fit a 3-tree forest, print each tree's root question and vote, then compare with a 100-tree forest. |

**🧠 ML concepts:** Bagging (bootstrap aggregation), Random feature subsets per split, Majority vote / probability averaging, Feature importance  
**🐍 Python concepts:** Dictionary of lists, list(dict) to get keys, np.column_stack, enumerate in a for loop, sorted with reverse=True, String repetition for ASCII bars

In [ ]:
# CONCEPT: a hand-built "forest" of 3 tiny trees votes on whether an auto claim goes to SIU
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# 🐍 PY: a dict of lists - each key is a column, each position is one claim
claims = {
    "days_policy_to_claim": [400, 20, 900, 15, 650, 30, 1200, 10, 300, 45, 800, 25, 500, 12],
    "claim_amount_k":       [3.2, 18.0, 2.1, 22.5, 4.0, 15.0, 5.5, 25.0, 2.8, 4.1, 6.0, 19.5, 3.5, 3.0],
    "police_report":        [1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1],
    "witnesses":            [2, 0, 1, 0, 1, 0, 2, 0, 1, 1, 0, 1, 2, 1],
}
is_fraud = np.array([0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0])
# 🐍 PY: list(claims) gives the keys; np.column_stack glues the lists side by side as columns
names = list(claims)
X = np.column_stack([claims[k] for k in names])
print("Claims:", X.shape[0], "| known fraud:", is_fraud.sum())

# 🧠 ML: Random Forest = many decision trees, each trained on a random resample of rows (bagging)
# 🧠 ML: max_features=2 means each split only looks at 2 random columns -> trees disagree usefully
rf = RandomForestClassifier(n_estimators=3, max_depth=2, max_features=2, random_state=42)
rf.fit(X, is_fraud)

new_claim = np.array([[18, 21.0, 0, 0]])  # new policy, big claim, no police report, no witnesses
print("\nEach tree's vote on the new claim:")
# 🐍 PY: enumerate gives a counter (i) alongside each item (tree) in the loop
for i, tree in enumerate(rf.estimators_):
    # 🧠 ML: each tree saw a different bootstrap sample, so it may learn a different rule
    vote = tree.predict(new_claim)[0]
    root = names[tree.tree_.feature[0]]
    print(f"  tree {i + 1}: first question asks about '{root}' -> vote {'FRAUD' if vote else 'ok'}")

# 🧠 ML: the forest averages the trees' probabilities - the 'wisdom of the crowd'
p = rf.predict_proba(new_claim)[0, 1]
print(f"\nForest fraud probability: {p:.2f} -> {'Refer to SIU' if p >= 0.5 else 'Fast-track payment'}")

# Now a bigger forest of 100 trees for a steadier answer
rf100 = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42).fit(X, is_fraud)
honest_claim = np.array([[700, 3.0, 1, 2]])
print(f"100-tree forest: suspicious claim {rf100.predict_proba(new_claim)[0, 1]:.2f}, "
      f"routine claim {rf100.predict_proba(honest_claim)[0, 1]:.2f}")

# 🧠 ML: feature importance averages how much each column cleaned up the splits across all trees
print("\nWhat the forest relies on:")
for n, imp in sorted(zip(names, rf100.feature_importances_), key=lambda t: t[1], reverse=True):
    print(f"  {n:<22} {'*' * int(imp * 30)} {imp:.2f}")

**Try this next:**
- Change max_features to 4 so every tree sees all columns - do the trees still disagree?
- Flip the police_report of the new claim to 1 and see how the probability changes.
- Add a 15th claim that is fraud but looks routine and see whether the forest learns it.

## BUILD: Rank 4,000 claims for SIU review
Train a class-weighted Random Forest on synthetic auto claims, use out-of-bag validation and measure lift for SIU capacity.

| 5W + H | |
|---|---|
| **Who** | A claims analytics team supporting the SIU, which can only investigate a small share of claims. |
| **What** | A fraud score for every claim, a lift table for the top 5/10/20%, and the main red-flag drivers. |
| **When** | Nightly on newly reported claims, or at FNOL once enough fields are captured. |
| **Where** | In the claims data warehouse; scores are pushed to the SIU work queue. |
| **Why** | Only ~5% of claims are fraudulent; random review wastes investigator time, so ranking matters more than a yes/no. |
| **How** | Simulate 4,000 claims with hidden red-flag patterns, fit RandomForestClassifier with balanced_subsample and oob_score, then ROC-AUC, classification report and lift. |

**🧠 ML concepts:** Class weighting for imbalance, Out-of-bag (OOB) validation, min_samples_leaf to limit overfitting, Precision and recall, Lift and capacity-based ranking, Impurity-based importance bias  
**🐍 Python concepts:** pandas boolean conditions with &, DataFrame.drop(columns=...), argsort and [::-1] reversal, Slicing the top k, pd.Series sort_values, DataFrame.query

In [ ]:
# BUILD: Random Forest on 4,000 synthetic auto claims to prioritise SIU referrals
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

rng = np.random.default_rng(42)
n = 4000
df = pd.DataFrame({
    "days_policy_to_claim": rng.integers(1, 1500, n),
    "claim_amount": rng.lognormal(8.6, 0.8, n).round(0),
    "police_report": rng.binomial(1, 0.65, n),
    "num_witnesses": rng.poisson(0.9, n),
    "prior_claims_3y": rng.poisson(0.4, n),
    "injury_claim": rng.binomial(1, 0.25, n),
    "vehicle_age_yrs": rng.integers(0, 20, n),
    "attorney_involved": rng.binomial(1, 0.15, n),
})
# Hidden fraud rule set: brand-new policy + injury claim, or no report + no witness + big amount
pattern1 = (df.days_policy_to_claim < 120) & (df.injury_claim == 1)
pattern2 = (df.police_report == 0) & (df.num_witnesses == 0) & (df.claim_amount > 7000)
logit = (-4.2 + 3.0 * pattern1 + 3.0 * pattern2 + 1.2 * df.attorney_involved
         + 0.5 * df.prior_claims_3y + 0.02 * df.vehicle_age_yrs)
df["siu_confirmed_fraud"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
print(f"Claims: {n}  confirmed fraud rate: {df.siu_confirmed_fraud.mean():.1%}")

# 🐍 PY: drop(columns=...) returns a new DataFrame without the label - the original df is unchanged
X = df.drop(columns="siu_confirmed_fraud")
y = df["siu_confirmed_fraud"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# 🧠 ML: class_weight='balanced_subsample' re-weights rare fraud inside every bootstrap sample
# 🧠 ML: min_samples_leaf=10 stops trees memorising single claims (less overfitting)
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=10, max_features="sqrt",
                            class_weight="balanced_subsample", oob_score=True,
                            n_jobs=2, random_state=42)
rf.fit(X_tr, y_tr)
# 🧠 ML: out-of-bag score = accuracy on rows each tree did NOT see - a free built-in validation
print(f"Out-of-bag accuracy: {rf.oob_score_:.3f}")

proba = rf.predict_proba(X_te)[:, 1]
print(f"Test ROC-AUC: {roc_auc_score(y_te, proba):.3f}")
# 🧠 ML: precision = share of referrals that are fraud; recall = share of all fraud we referred
print(classification_report(y_te, (proba >= 0.5).astype(int), target_names=["legit", "fraud"], digits=2))

# 🧠 ML: SIU has limited investigators - rank claims by score and check the top slice (lift)
# 🐍 PY: argsort()[::-1] gives indices from highest to lowest score; slicing takes the top k
order = np.argsort(proba)[::-1]
base_rate = y_te.mean()
print("SIU capacity | fraud found | hit rate | lift vs random")
for pct in [0.05, 0.10, 0.20]:
    k = int(len(order) * pct)
    hits = y_te.values[order[:k]].sum()
    print(f"top {pct:>4.0%}     | {hits:>11} | {hits / k:>8.1%} | {hits / k / base_rate:>6.1f}x")

# 🧠 ML: impurity importance favours many-valued columns (vehicle_age) - verify with permutation importance
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop red-flag drivers:")
for col, v in imp.head(5).items():
    print(f"  {col:<22} {'#' * int(v * 60)} {v:.3f}")

k = int(len(order) * 0.10)
# 🐍 PY: .loc selects rows by index label; .query filters with a readable string expression
saved = df.loc[X_te.index[order[:k]]].query("siu_confirmed_fraud == 1").claim_amount.sum()
print(f"\nInvestigating top 10% ({k} claims) targets ${saved:,.0f} of fraudulent payouts")

**Try this next:**
- Set class_weight=None and compare recall for the fraud class.
- Change min_samples_leaf to 1 and compare OOB accuracy with test AUC - spot the overfitting.
- Add a feature for claims reported on a Monday and see whether it shows up in importance.

## PRODUCTION: SIU referral engine with capacity threshold
Build a production Random Forest pipeline with imputation and encoding, tuned by CV, compared with a baseline, thresholded by SIU capacity and checked for fairness.

| 5W + H | |
|---|---|
| **Who** | Claims analytics, SIU leadership, and the insurer's model governance / compliance team. |
| **What** | A saved pipeline that scores every claim, a capacity-based referral threshold, net savings estimate and permutation importance. |
| **When** | Retrained quarterly; scores run at FNOL and again when new claim facts arrive. |
| **Where** | Training on the analytics platform; the joblib model is called by the claims system's triage rules. |
| **Why** | Real claims data is messy (missing values, categories), SIU has fixed capacity, and NAIC and state rules require governed, non-discriminatory use. |
| **How** | ColumnTransformer (median impute + one-hot) + Random Forest, RandomizedSearchCV on PR-AUC, LogReg baseline, top-8% threshold, cost/benefit, permutation importance, state referral check, joblib. |

**🧠 ML concepts:** Imputation inside the pipeline, Hyperparameter tuning (depth, leaf size, max_features), PR-AUC for rare events, Capacity-based thresholds, Permutation vs impurity importance, Fairness and governance (NAIC AI Bulletin)  
**🐍 Python concepts:** Functions with keyword defaults, Generators with yield, Conditional list building, Tuple unpacking of constants, groupby on an aligned Series, Dict bundling model + metadata

In [ ]:
# PRODUCTION: production SIU referral model - Random Forest pipeline, CV tuning, capacity-based threshold
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
NUM = ["days_policy_to_claim", "claim_amount", "num_witnesses", "prior_claims_3y",
       "vehicle_age_yrs", "days_to_report", "repair_estimate_ratio"]
CAT = ["loss_type", "repair_shop_tier", "state"]
BIN = ["police_report", "injury_claim", "attorney_involved"]
# 🐍 PY: tuple unpacking assigns two constants in one line
SIU_COST, SIU_CAPACITY = 450, 0.08   # $ per investigation; SIU can review 8% of claims


def make_claims(n=8000, seed=42):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "days_policy_to_claim": rng.integers(1, 1500, n),
        "claim_amount": rng.lognormal(8.6, 0.8, n).round(0),
        "num_witnesses": rng.poisson(0.9, n),
        "prior_claims_3y": rng.poisson(0.4, n),
        "vehicle_age_yrs": rng.integers(0, 20, n),
        "days_to_report": rng.exponential(4, n).round(),
        "repair_estimate_ratio": rng.lognormal(0, 0.3, n).round(2),  # estimate / book repair cost
        "loss_type": rng.choice(["rear_end", "theft", "single_vehicle", "parking", "hail"], n),
        "repair_shop_tier": rng.choice(["network", "independent", "flagged"], n, p=[.6, .35, .05]),
        "state": rng.choice(["TX", "FL", "OK", "CA", "NY"], n),
        "police_report": rng.binomial(1, 0.65, n),
        "injury_claim": rng.binomial(1, 0.25, n),
        "attorney_involved": rng.binomial(1, 0.15, n),
    })
    # Real data has gaps: 5% of repair estimates missing
    # 🐍 PY: .loc[mask, col] = value overwrites only the rows where the mask is True
    df.loc[rng.random(n) < 0.05, "repair_estimate_ratio"] = np.nan
    p1 = (df.days_policy_to_claim < 120) & (df.injury_claim == 1)
    p2 = (df.police_report == 0) & (df.num_witnesses == 0) & (df.claim_amount > 7000)
    p3 = (df.repair_shop_tier == "flagged") & (df.repair_estimate_ratio.fillna(1) > 1.3)
    logit = (-4.6 + 3.6 * p1 + 3.6 * p2 + 3.8 * p3 + 0.6 * df.attorney_involved
             + 0.3 * df.prior_claims_3y + 0.04 * df.days_to_report + 0.5 * (df.loss_type == "theft"))
    df["fraud"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    return df


def preprocessor(scale=False):
    # 🧠 ML: trees need no scaling, but missing values must be imputed and categories one-hot encoded
    num_steps = [("imp", SimpleImputer(strategy="median"))] + ([("sc", StandardScaler())] if scale else [])
    return ColumnTransformer([
        ("num", Pipeline(num_steps), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
        ("bin", "passthrough", BIN),
    ])


def lift_table(y, score, cuts=(0.02, 0.05, 0.08, 0.15)):
    order = np.argsort(-score)
    for c in cuts:
        k = int(len(y) * c)
        hits = y[order[:k]].sum()
        # 🐍 PY: yield turns this function into a generator - rows are produced one at a time
        yield c, k, hits, hits / max(y.sum(), 1)


def main():
    df = make_claims()
    X, y = df.drop(columns="fraud"), df.fraud
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    print(f"Claims {len(df):,} | fraud rate {y.mean():.1%} | train {len(X_tr)} / test {len(X_te)}")

    # 🧠 ML: baseline logistic regression - if the forest cannot beat this, prefer the simpler model
    base = Pipeline([("pre", preprocessor(scale=True)),
                     ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))]).fit(X_tr, y_tr)

    rf = Pipeline([("pre", preprocessor()),
                   ("clf", RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample",
                                                  n_jobs=2, random_state=42))])
    # 🧠 ML: tune tree depth, leaf size and features-per-split; these control the bias/variance trade-off
    space = {"clf__max_depth": [6, 10, None], "clf__min_samples_leaf": [3, 10, 25],
             "clf__max_features": ["sqrt", 0.5]}
    search = RandomizedSearchCV(rf, space, n_iter=5, scoring="average_precision", random_state=42,
                                cv=StratifiedKFold(3, shuffle=True, random_state=42), n_jobs=1)
    search.fit(X_tr, y_tr)
    model = search.best_estimator_
    print("Best RF params:", {k.split("__")[1]: v for k, v in search.best_params_.items()})

    print("\nModel                 ROC-AUC  PR-AUC")
    scores = {}
    for name, m in [("LogReg baseline", base), ("Random Forest", model)]:
        scores[name] = m.predict_proba(X_te)[:, 1]
        print(f"{name:<20} {roc_auc_score(y_te, scores[name]):>8.3f} {average_precision_score(y_te, scores[name]):>7.3f}")

    # 🧠 ML: the threshold is set by SIU capacity, not 0.5 - refer the top-scoring 8% of claims
    p = scores["Random Forest"]
    thr = np.quantile(p, 1 - SIU_CAPACITY)
    print(f"\nCapacity threshold (top {SIU_CAPACITY:.0%}): score >= {thr:.3f}")
    print("Referred | claims | fraud caught | share of all fraud")
    for c, k, hits, share in lift_table(y_te.values, p):
        print(f"top {c:>4.0%} | {k:>6} | {hits:>12} | {share:>8.0%}")

    # 🧠 ML: translate model output into money - fraud payouts avoided minus investigation cost
    flagged = p >= thr
    fraud_paid_avoided = X_te.claim_amount[flagged & (y_te == 1)].sum()
    cost = flagged.sum() * SIU_COST
    print(f"Fraud payouts avoided ${fraud_paid_avoided:,.0f} - SIU cost ${cost:,.0f} "
          f"= net ${fraud_paid_avoided - cost:,.0f}")

    # 🧠 ML: permutation importance on raw columns is less biased than impurity importance
    pi = permutation_importance(model, X_te, y_te, scoring="average_precision", n_repeats=3,
                                random_state=42, n_jobs=1)
    imp = pd.Series(pi.importances_mean, index=X_te.columns).sort_values(ascending=False)
    print("\nPermutation importance (PR-AUC drop):")
    for col, v in imp.head(6).items():
        print(f"  {col:<22} {v:+.4f}")

    # 🧠 ML: fairness check - referral rates by state should be explainable by risk, not geography alone
    # 🐍 PY: groupby on another Series aligned by index, then mean() of booleans = referral rate
    rates = pd.Series(flagged, index=X_te.index).groupby(X_te.state).mean()
    print("\nReferral rate by state:", ", ".join(f"{s} {r:.0%}" for s, r in rates.items()))
    print("Governance: NAIC AI Model Bulletin + state unfair-claims laws -> model only PRIORITISES;")
    print("a human SIU investigator decides. Log reasons, monitor drift, re-validate yearly.")

    path = os.path.join(tempfile.mkdtemp(), "siu_rf.joblib")
    # 🐍 PY: a dict bundles the model with metadata so scoring code knows the threshold used
    joblib.dump({"model": model, "threshold": float(thr), "capacity": SIU_CAPACITY}, path)
    print(f"Saved {os.path.basename(path)} ({os.path.getsize(path) // 1024} KB)")


if __name__ == "__main__":
    main()

**Try this next:**
- Change SIU_CAPACITY to 0.05 and 0.15 and compare net savings - where are the diminishing returns?
- Add 'clf__n_estimators': [100, 300] to the search space and see if it helps.
- Remove the 'state' column and confirm the model performs the same (it should - state carries no fraud signal here).